# Palace Straight Dielectric Waveguide - Wave Ports

This notebook sets up a straight dielectric waveguide and runs a Palace driven simulation using waveport boundary conditions.

### Build straight waveguide geometry

In [8]:
from ubcpdk import PDK, cells

PDK.activate()

c = cells.straight(length=10.0)

cc = c.copy()
cc.draw_ports()
cc

PermissionError: [Errno 13] Permission denied: '/tmp/gdsfactory/layer_properties.lyp'

Component(name=straight_L10_CSstrip_WNone_N2$2, ports=['o1', 'o2'], pins=[], instances=[], locked=False, kcl=DEFAULT)

### Configure Palace driven simulation

In [9]:
from gsim.palace import DrivenSim

sim = DrivenSim()
sim.set_output_dir("./palace-sim-straight-waveport")
sim.set_geometry(c)

sim.set_stack(
    substrate_thickness=2.0,
    add_oxide_dielectric=False,
    add_passivation_dielectric=False,
)

sim.set_numerical(
    solver_type="MUMPS",
    max_iterations=1,
    tolerance=1e-8,
)

for i, port in enumerate(c.ports):
    sim.add_wave_port(
        str(port.name),
        layer="core",
        z_margin=1.5,
        lateral_margin=1.5,
        max_size=False,
        mode=1,
        excited=(i == 0),
    )

sim.set_driven(fmin=190e12, fmax=200e12, num_points=21)

print(sim.validate_config())

Validation: PASSED


### Generate mesh

In [ ]:
sim.set_airbox(
    margin_x=0.0,
    margin_y=2.0,
    margin_above=0.5,
    margin_below=0.5,
)

sim.mesh(
    preset="fine",
    refined_mesh_size=0.04,
    max_mesh_size=0.2,
    fmax=200e12,
)

Mesh has 233,705 nodes (>75,000) — simulation may be slow.


Mesh Summary
Dimensions: 10.0 x 4.5 x 6.5 µm
Nodes:      233,705
Elements:   1,428,340
Tetrahedra: 1,340,770
Edge length: 0.03 - 0.22 µm
Quality:    0.791 (min: 0.203)
SICN:       0.815 (all valid)
----------------------------------------
Volumes (2):
  - core [1]
  - airbox [2]
Surfaces (4):
  - P1 [3]
  - P2 [4]
  - airbox__core [5]
  - airbox__None [6]
----------------------------------------
Mesh:   palace-sim-straight-waveport/palace.msh

In [11]:
sim.plot_mesh(
    style="solid",
    transparent_groups=["airbox__None"],
    interactive=True,
)

Widget(value='<iframe src="http://localhost:42333/index.html?ui=P_0x7858a8181700_1&reconnect=auto" class="pyvi…

### Run simulation

In [ ]:
import json
from pathlib import Path

config_path = sim.write_config(photonic=True)
cfg = json.loads(Path(config_path).read_text())
print("Solver.Linear:", cfg["Solver"]["Linear"])

results = sim.run_local(num_processes=4, verbose=True)

Running Palace simulation in palace-sim-straight-waveport via Apptainer
Command: apptainer run /home/martin/Desktop/palace/Palace.sif -np 4 config.json
Processes: 4
>> /usr/lib64/mpich/bin/mpirun -n 4 /opt/palace/bin/palace-x86_64.bin config.json


Solver.Linear: {'Type': 'MUMPS', 'KSPType': 'GMRES', 'Tol': 1e-08, 'MaxIts': 1, 'MGMaxLevels': 1, 'EstimatorMaxIts': 0, 'EstimatorTol': 1e-06, 'DivFreeTol': 1e-06, 'DivFreeMaxIts': 0, 'PCMatReal': False, 'ComplexCoarseSolve': True}


_____________     _______
_____   __   \____ __   /____ ____________
____   /_/  /  __ ` /  /  __ ` /  ___/  _ \
___   _____/  /_/  /  /  /_/  /  /__/  ___/
  /__/     \___,__/__/\___,__/\_____\_____/
--> Warning!
Output folder is not empty; program will overwrite content! (output/palace)
Git changeset ID: v0.14.0-305-g51d61b03
Running with 4 MPI processes, 1 OpenMP thread
Device configuration: omp,cpu
Memory configuration: host-std
libCEED backend: /cpu/self/xsmm/blocked
Finished partitioning mesh into 4 subdomains
Characteristic length and time scales:
 L₀ = 1.000e-05 m, t₀ = 3.336e-05 ns
Mesh curvature order: 1
Mesh bounding box:
 (Xmin, Ymin, Zmin) = (+0.000e+00, -2.250e-06, -3.500e-06) m
 (Xmax, Ymax, Zmax) = (+1.000e-05, +2.250e-06, +3.000e-06) m
Parallel Mesh Stats:
                minimum     average     maximum       total
 vertices         56683       58426       60268      233705
 edges           398079      403259      408687     1613037
 faces           676591      680025 

### Plot S-parameters

In [ ]:
results.plot_interactive()

Port mapping: Port 1: p1, Port 2: p2


In [ ]:
results.plot_interactive(phase=True)

Port mapping: Port 1: p1, Port 2: p2
